# Advanced Reporting & Quality Analysis

Bu notebook tüm run'lardan quality metrics ve reproducibility checks rapor eder.
- Fallback rate trends
- Sample validation error distribution
- Seed reproducibility validation
- Multi-run comparison heatmaps

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

sns.set_theme()
%matplotlib inline

RESULTS_DIR = Path('results')

In [ ]:
# Load all run metadata
runs_metadata = []
runs_stats = []

for run_dir in sorted(RESULTS_DIR.glob('*')):
    if not run_dir.is_dir():
        continue
    
    config_file = run_dir / 'config_snapshot.json'
    stats_file = run_dir / 'stats.json'
    env_file = run_dir / 'environment.json'
    
    if not config_file.exists():
        continue
    
    try:
        config = json.loads(config_file.read_text())
        stats = json.loads(stats_file.read_text()) if stats_file.exists() else {}
        env = json.loads(env_file.read_text()) if env_file.exists() else {}
        
        runs_metadata.append({
            'run_name': run_dir.name,
            'timestamp': config.get('timestamp'),
            'seed': config.get('seed'),
            'models': config.get('models', []),
            'datasets': config.get('datasets', []),
            'num_samples': config.get('num_samples'),
        })
        
        stats_data = stats.get('statistics', {})
        runs_stats.append({
            'run_name': run_dir.name,
            'total_samples': stats_data.get('total_samples'),
            'fallback_rate_avg': stats_data.get('_meta_fallback_rates', {}).get('all_evaluators_fallback_rate'),
            'seed': config.get('seed'),
        })
    except Exception as e:
        print(f'Error loading {run_dir.name}: {e}')

runs_df = pd.DataFrame(runs_metadata)
stats_df = pd.DataFrame(runs_stats)

print(f'Loaded {len(runs_df)} runs')
display(runs_df.head())

In [ ]:
# Quality Summary Report
print('=== QUALITY SUMMARY ===\n')

if len(stats_df) > 0:
    print('Fallback Rate Statistics:')
    print(f'  Mean: {stats_df["fallback_rate_avg"].mean():.3f}')
    print(f'  Std: {stats_df["fallback_rate_avg"].std():.3f}')
    print(f'  Min: {stats_df["fallback_rate_avg"].min():.3f}')
    print(f'  Max: {stats_df["fallback_rate_avg"].max():.3f}')
    
    print('\nSeed Consistency:')
    seed_counts = stats_df.groupby('seed').size()
    for seed, count in seed_counts.items():
        if seed is not None:
            print(f'  Seed {seed}: {count} runs')
    
    if seed_counts.max() > 1:
        print('  ✓ Reproducible runs detected (same seed in multiple runs)')
    else:
        print('  ⚠️  No reproducible runs (each seed used once)')
        
    print('\nSample Coverage:')
    total_samples_all = stats_df['total_samples'].sum()
    print(f'  Total samples across all runs: {int(total_samples_all) if pd.notna(total_samples_all) else "N/A"}')

In [ ]:
# Error Analysis
print('=== ERROR ANALYSIS ===\n')

all_errors = []
for run_dir in RESULTS_DIR.glob('*/errors.jsonl'):
    run_name = run_dir.parent.name
    for line in run_dir.read_text().splitlines():
        if line.strip():
            err = json.loads(line)
            err['run_name'] = run_name
            all_errors.append(err)

if all_errors:
    error_df = pd.DataFrame(all_errors)
    print('Error type distribution across all runs:')
    error_counts = error_df.groupby('error_type').size().sort_values(ascending=False)
    display(error_counts.to_frame('count'))
    
    print('\nError type by run:')
    error_by_run = error_df.groupby(['run_name', 'error_type']).size().unstack(fill_value=0)
    display(error_by_run)
else:
    print('No errors logged across all runs - ✓ All samples processed successfully')

In [ ]:
# Reproducibility Check Report
print('=== REPRODUCIBILITY VALIDATION ===\n')

reproducible_runs = stats_df.groupby('seed').filter(lambda x: len(x) > 1)
if len(reproducible_runs) > 0:
    print(f'Found {len(reproducible_runs)} samples from reproducible runs (shared seeds)\n')
    for seed, group in reproducible_runs.groupby('seed'):
        print(f'Seed {seed}:')
        for _, row in group.iterrows():
            print(f'  - {row["run_name"]} (fallback_rate: {row["fallback_rate_avg"]:.3f})')
        print()
else:
    print('No reproducible runs found yet.')
    print('To enable reproducibility validation:',)
    print('  1. Run same model/dataset with --seed <N>',)
    print('  2. Compare fallback rates and metrics across runs')

In [ ]:
# Visualization: Fallback Rate Trend
if len(stats_df) > 1:
    fig, ax = plt.subplots(figsize=(12, 5))
    stats_df_sorted = stats_df.sort_values('run_name')
    ax.plot(range(len(stats_df_sorted)), stats_df_sorted['fallback_rate_avg'], marker='o', linestyle='-')
    ax.set_xlabel('Run (chronological)')
    ax.set_ylabel('Fallback Rate (avg)')
    ax.set_title('Fallback Rate Trend Across Runs')
    ax.set_xticks(range(len(stats_df_sorted)))
    ax.set_xticklabels(stats_df_sorted['run_name'], rotation=45, ha='right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print('Trend: Lower fallback rates indicate more reliable metric computation')